# Model 2 — EfficientNet-B0 Readiness Prediction

This notebook contains only the main **readiness-only Model 2** workflow:

1. Load the existing dataset and targets.
2. Prepare train/validation/test splits and data loaders.
3. Build and train the EfficientNet-B0 model to predict `R_seg` and `R_F1`.
4. Evaluate on the held-out test fold.
5. Demonstrate one test image with the input, ground-truth targets, model predictions, and Grad-CAM.

The notebook does not include the earlier sanity-check, model-comparison, archival, or Model 3 experiment cells. Training runs from scratch when the training cell is executed.


## 1. Setup

In [ ]:
# ============================================================
# CELL 1 — SETUP
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import json
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = "/content/drive/MyDrive/AI_Readiness_Project"
DATASET_ROOT = os.path.join(
    PROJECT_ROOT,
    "AI_Readiness_Dataset"
)

TARGET_FILE = os.path.join(
    DATASET_ROOT,
    "Targets",
    "AI_Readiness_Targets_CORRECTED.csv"
)

METADATA_FILE = os.path.join(
    DATASET_ROOT,
    "metadata.csv"
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

if DEVICE.type != "cuda":
    raise RuntimeError("GPU not available.")

print("GPU:", torch.cuda.get_device_name(0))

ValueError: mount failed

## 2. Load targets and build the dataset

In [ ]:
# ============================================================
# CELL 2 — BUILD FINAL DATASET
# ============================================================

targets = pd.read_csv(
    TARGET_FILE
)

metadata = pd.read_csv(
    METADATA_FILE
)

targets["image_id"] = (
    targets["image_id"]
    .astype(str)
    .str.zfill(5)
)

metadata["image_id"] = (
    metadata["image_name"]
    .str.replace(".png", "", regex=False)
    .astype(str)
    .str.zfill(5)
)

df = metadata.merge(
    targets,
    on="image_id",
    how="inner"
)

def abs_path(x):
    if os.path.isabs(str(x)):
        return str(x)
    return os.path.join(
        DATASET_ROOT,
        str(x)
    )

df["degraded_path"] = (
    df["degraded_path"]
    .apply(abs_path)
)

PRIMARY_TARGETS = [
    "R_seg",
    "R_F1"
]

AUX_TARGETS = [
    "blur_score",
    "motion_score",
    "brightness_score",
    "contrast_score",
    "noise_score",
    "jpeg_score",
    "stain_score",
    "tissue_score",
    "bubble_score",
    "vignette_score"
]

df = (
    df
    .sort_values(
        "image_id",
        key=lambda x: x.astype(int)
    )
    .reset_index(drop=True)
)

print("Rows:", len(df))
print("Unique IDs:", df["image_id"].nunique())
print(df["split"].value_counts())

## 3. Assign PanNuke folds

In [ ]:
# ============================================================
# CELL — ASSIGN PANNUKE FOLD SPLITS
# Fold 1 = Train
# Fold 2 = Validation
# Fold 3 = Test
# ============================================================

print("=" * 80)
print("ASSIGNING PANNUKE FOLD SPLITS")
print("=" * 80)

# PanNuke fold sizes verified earlier
FOLD1_SIZE = 2656
FOLD2_SIZE = 2523
FOLD3_SIZE = 2722

TOTAL_SIZE = (
    FOLD1_SIZE
    + FOLD2_SIZE
    + FOLD3_SIZE
)

print(
    "Expected PanNuke images:",
    TOTAL_SIZE
)

# ------------------------------------------------------------
# Create ordered list of all 7901 image IDs
# ------------------------------------------------------------

all_ids = (
    metadata
    .sort_values(
        "image_id",
        key=lambda x: x.astype(int)
    )["image_id"]
    .tolist()
)

if len(all_ids) != TOTAL_SIZE:

    raise ValueError(
        f"Expected {TOTAL_SIZE} ordered images, "
        f"but found {len(all_ids)}"
    )

# ------------------------------------------------------------
# Assign folds by PanNuke ordering
# ------------------------------------------------------------

fold_labels = {}

for i, image_id in enumerate(all_ids):

    if i < FOLD1_SIZE:

        fold_labels[image_id] = "fold1"

    elif i < (
        FOLD1_SIZE
        +
        FOLD2_SIZE
    ):

        fold_labels[image_id] = "fold2"

    else:

        fold_labels[image_id] = "fold3"


# Add fold column
df["fold"] = (
    df["image_id"]
    .map(fold_labels)
)

# ------------------------------------------------------------
# Map to ML split
# ------------------------------------------------------------

df["split"] = df["fold"].map({
    "fold1": "train",
    "fold2": "validation",
    "fold3": "test"
})

print("\nFold distribution:")
print(
    df["fold"].value_counts()
    .sort_index()
)

print("\nFinal usable split:")
print(
    df["split"].value_counts()
)

# ------------------------------------------------------------
# Check expected counts after removing 81 images
# ------------------------------------------------------------

print("\nFold counts by final 7,820-image dataset:")

for fold in [
    "fold1",
    "fold2",
    "fold3"
]:

    count = (
        df["fold"] == fold
    ).sum()

    print(
        f"{fold}: {count}"
    )

print("\n✓ PanNuke fold assignment complete")
print("=" * 80)

## 4. Normalize auxiliary targets

In [ ]:
# ============================================================
# CELL 3 — TARGET NORMALIZATION
# ============================================================

train_df = df[
    df["split"] == "train"
].copy()

scalers = {}

for col in AUX_TARGETS:

    mn = float(
        train_df[col].min()
    )

    mx = float(
        train_df[col].max()
    )

    scalers[col] = {
        "min": mn,
        "max": mx
    }

    if mx > mn:

        df[f"{col}_norm"] = (
            df[col] - mn
        ) / (
            mx - mn
        )

    else:

        df[f"{col}_norm"] = 0.0

df["R_seg_norm"] = df["R_seg"].astype(float)
df["R_F1_norm"] = df["R_F1"].astype(float)

SCALER_PATH = os.path.join(
    DATASET_ROOT,
    "Targets",
    "EfficientNet_Rseg_RF1_scalers.json"
)

with open(SCALER_PATH, "w") as f:
    json.dump(
        scalers,
        f,
        indent=4
    )

print("✓ Targets prepared")

## 5. Dataset and data loaders

In [ ]:
# ============================================================
# CELL 4 — DATASET + DATALOADERS
# ============================================================

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

MEAN = [
    0.485,
    0.456,
    0.406
]

STD = [
    0.229,
    0.224,
    0.225
]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])


class ReadinessDataset(Dataset):

    def __init__(self, data, transform):
        self.data = data.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        image = Image.open(
            row["degraded_path"]
        ).convert("RGB")

        image = self.transform(image)

        readiness = torch.tensor(
            [
                row["R_seg_norm"],
                row["R_F1_norm"]
            ],
            dtype=torch.float32
        )

        degradation = torch.tensor(
            [
                row[f"{c}_norm"]
                for c in AUX_TARGETS
            ],
            dtype=torch.float32
        )

        return {
            "image": image,
            "readiness": readiness,
            "degradation": degradation,
            "image_id": row["image_id"]
        }


train_data = df[df["split"] == "train"]
val_data = df[df["split"] == "validation"]
test_data = df[df["split"] == "test"]

train_dataset = ReadinessDataset(
    train_data,
    train_transform
)

val_dataset = ReadinessDataset(
    val_data,
    eval_transform
)

test_dataset = ReadinessDataset(
    test_data,
    eval_transform
)

BATCH_SIZE = 64
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

## 6. Build Model 2 — readiness-only EfficientNet-B0

In [ ]:
# ============================================================
# READINESS-ONLY EFFICIENTNET-B0
# Outputs ONLY:
#   1. R_seg
#   2. R_F1
# ============================================================

import torch
import torch.nn as nn
from torchvision.models import (
    efficientnet_b0,
    EfficientNet_B0_Weights
)

print("=" * 80)
print("BUILDING READINESS-ONLY EFFICIENTNET-B0")
print("=" * 80)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

if DEVICE.type != "cuda":
    raise RuntimeError("GPU required.")

backbone = efficientnet_b0(
    weights=EfficientNet_B0_Weights.DEFAULT
)

feature_dim = (
    backbone.classifier[1].in_features
)

backbone.classifier = nn.Identity()


class ReadinessOnlyEfficientNet(
    nn.Module
):

    def __init__(
        self,
        backbone,
        feature_dim
    ):

        super().__init__()

        self.backbone = backbone

        self.head = nn.Sequential(
            nn.Linear(
                feature_dim,
                256
            ),
            nn.BatchNorm1d(256),
            nn.SiLU(),
            nn.Dropout(0.25),
            nn.Linear(
                256,
                2
            )
        )

    def forward(self, x):

        features = self.backbone(x)

        readiness = self.head(
            features
        )

        return readiness


model = ReadinessOnlyEfficientNet(
    backbone,
    feature_dim
).to(DEVICE)

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

print(
    "Feature dimension:",
    feature_dim
)

print(
    "Parameters:",
    sum(
        p.numel()
        for p in model.parameters()
    )
)

print("✓ Readiness-only model ready")

## 7. Train Model 2

In [ ]:
# ============================================================
# READINESS-ONLY 3-STAGE TRAINING
# ============================================================

import os
import time
import pandas as pd
import torch.nn as nn

from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm

print("=" * 80)
print("READINESS-ONLY TRAINING")
print("=" * 80)

MODEL_DIR = os.path.join(
    PROJECT_ROOT,
    "Models",
    "EfficientNet_Rseg_RF1_ReadinessOnly"
)

CHECKPOINT_DIR = os.path.join(
    MODEL_DIR,
    "checkpoints"
)

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

BEST_MODEL_PATH = os.path.join(
    CHECKPOINT_DIR,
    "best_model.pth"
)

LAST_MODEL_PATH = os.path.join(
    CHECKPOINT_DIR,
    "last_checkpoint.pth"
)

HISTORY_PATH = os.path.join(
    MODEL_DIR,
    "training_history.csv"
)

STAGE1_EPOCHS = 3
STAGE2_EPOCHS = 5
STAGE3_EPOCHS = 12

READINESS_WEIGHT = 1.0

WEIGHT_DECAY = 1e-4
PATIENCE = 4

criterion = nn.SmoothL1Loss()


def readiness_loss(
    prediction,
    target
):

    rseg_loss = criterion(
        prediction[:, 0],
        target[:, 0]
    )

    rf1_loss = criterion(
        prediction[:, 1],
        target[:, 1]
    )

    total = (
        rseg_loss
        +
        rf1_loss
    ) / 2.0

    return {
        "total": total,
        "R_seg": rseg_loss,
        "R_F1": rf1_loss
    }


def set_stage(stage):

    # --------------------------------------------------------
    # Stage 1 — frozen backbone
    # --------------------------------------------------------

    if stage == 1:

        for p in model.backbone.parameters():
            p.requires_grad = False

        print(
            "\nStage 1: backbone frozen"
        )

        return [
            {
                "params":
                    model.head.parameters(),
                "lr": 1e-3
            }
        ]

    # --------------------------------------------------------
    # Stage 2 — final blocks
    # --------------------------------------------------------

    if stage == 2:

        for p in model.backbone.parameters():
            p.requires_grad = False

        for block in model.backbone.features[-3:]:

            for p in block.parameters():
                p.requires_grad = True

        print(
            "\nStage 2: final EfficientNet blocks unfrozen"
        )

        return [
            {
                "params":
                    [
                        p
                        for p in model.backbone.parameters()
                        if p.requires_grad
                    ],
                "lr": 2e-5
            },
            {
                "params":
                    model.head.parameters(),
                "lr": 2e-4
            }
        ]

    # --------------------------------------------------------
    # Stage 3 — full fine-tuning
    # --------------------------------------------------------

    if stage == 3:

        for p in model.backbone.parameters():
            p.requires_grad = True

        print(
            "\nStage 3: full EfficientNet fine-tuning"
        )

        return [
            {
                "params":
                    model.backbone.parameters(),
                "lr": 1e-5
            },
            {
                "params":
                    model.head.parameters(),
                "lr": 1e-4
            }
        ]


def train_epoch(
    optimizer,
    scaler
):

    model.train()

    sums = {
        "total": 0.0,
        "R_seg": 0.0,
        "R_F1": 0.0
    }

    total_samples = 0

    for batch in tqdm(
        train_loader,
        desc="Training",
        leave=False
    ):

        images = batch[
            "image"
        ].to(
            DEVICE,
            non_blocking=True
        )

        target = batch[
            "readiness"
        ].to(
            DEVICE,
            non_blocking=True
        )

        n = images.size(0)

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            prediction = model(
                images
            )

            losses = readiness_loss(
                prediction,
                target
            )

        scaler.scale(
            losses["total"]
        ).backward()

        scaler.step(
            optimizer
        )

        scaler.update()

        for key in sums:

            sums[key] += (
                losses[key].item()
                * n
            )

        total_samples += n

    return {
        key:
            value / total_samples
        for key, value in sums.items()
    }


@torch.no_grad()
def validate_epoch():

    model.eval()

    sums = {
        "total": 0.0,
        "R_seg": 0.0,
        "R_F1": 0.0
    }

    total_samples = 0

    for batch in tqdm(
        val_loader,
        desc="Validation",
        leave=False
    ):

        images = batch[
            "image"
        ].to(
            DEVICE,
            non_blocking=True
        )

        target = batch[
            "readiness"
        ].to(
            DEVICE,
            non_blocking=True
        )

        n = images.size(0)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            prediction = model(
                images
            )

            losses = readiness_loss(
                prediction,
                target
            )

        for key in sums:

            sums[key] += (
                losses[key].item()
                * n
            )

        total_samples += n

    return {
        key:
            value / total_samples
        for key, value in sums.items()
    }


scaler = torch.amp.GradScaler(
    "cuda"
)

history = []

best_val = float("inf")
global_epoch = 0

for stage, epochs in [
    (1, STAGE1_EPOCHS),
    (2, STAGE2_EPOCHS),
    (3, STAGE3_EPOCHS)
]:

    parameter_groups = set_stage(
        stage
    )

    optimizer = AdamW(
        parameter_groups,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2
    )

    no_improve = 0

    for local_epoch in range(
        epochs
    ):

        global_epoch += 1

        start = time.time()

        train_metrics = train_epoch(
            optimizer,
            scaler
        )

        val_metrics = validate_epoch()

        scheduler.step(
            val_metrics["total"]
        )

        elapsed = time.time() - start

        record = {
            "stage": stage,
            "epoch": global_epoch,

            "train_loss":
                train_metrics["total"],

            "train_R_seg":
                train_metrics["R_seg"],

            "train_R_F1":
                train_metrics["R_F1"],

            "val_loss":
                val_metrics["total"],

            "val_R_seg":
                val_metrics["R_seg"],

            "val_R_F1":
                val_metrics["R_F1"],

            "learning_rate":
                optimizer.param_groups[0]["lr"],

            "time_sec": elapsed
        }

        history.append(
            record
        )

        pd.DataFrame(
            history
        ).to_csv(
            HISTORY_PATH,
            index=False
        )

        current_val = (
            val_metrics["total"]
        )

        if current_val < best_val:

            best_val = current_val
            no_improve = 0

            torch.save(
                {
                    "stage": stage,
                    "epoch": global_epoch,
                    "model_state":
                        model.state_dict(),
                    "best_val":
                        best_val,
                    "history":
                        history
                },
                BEST_MODEL_PATH
            )

            marker = " ★ BEST"

        else:

            no_improve += 1
            marker = ""

        torch.save(
            {
                "stage": stage,
                "epoch": global_epoch,
                "model_state":
                    model.state_dict(),
                "optimizer_state":
                    optimizer.state_dict(),
                "scheduler_state":
                    scheduler.state_dict(),
                "scaler_state":
                    scaler.state_dict(),
                "best_val":
                    best_val,
                "history":
                    history
            },
            LAST_MODEL_PATH
        )

        print(
            f"\nStage {stage} | "
            f"Epoch {global_epoch}"
        )

        print(
            f"Train loss : "
            f"{train_metrics['total']:.6f}"
        )

        print(
            f"Val loss   : "
            f"{val_metrics['total']:.6f}"
        )

        print(
            f"Val R_seg  : "
            f"{val_metrics['R_seg']:.6f}"
        )

        print(
            f"Val R_F1   : "
            f"{val_metrics['R_F1']:.6f}"
        )

        print(
            f"Time       : "
            f"{elapsed / 60:.2f} min"
            f"{marker}"
        )

        if no_improve >= PATIENCE:
            print(
                "Early stopping this stage."
            )
            break

print("\n" + "=" * 80)
print("READINESS-ONLY TRAINING COMPLETE")
print("=" * 80)

print(
    "Best validation loss:",
    best_val
)

print(
    "Best model:",
    BEST_MODEL_PATH
)

print("=" * 80)

## 8. Evaluate on the held-out test fold

In [ ]:
# ============================================================
# READINESS-ONLY — FINAL FOLD 3 TEST EVALUATION
# ============================================================

import os
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from scipy.stats import pearsonr

print("=" * 80)
print("READINESS-ONLY — FOLD 3 TEST EVALUATION")
print("=" * 80)

# ============================================================
# PATHS
# ============================================================

MODEL_DIR = os.path.join(
    PROJECT_ROOT,
    "Models",
    "EfficientNet_Rseg_RF1_ReadinessOnly"
)

BEST_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "checkpoints",
    "best_model.pth"
)

METRICS_PATH = os.path.join(
    MODEL_DIR,
    "fold3_test_metrics.csv"
)

PREDICTIONS_PATH = os.path.join(
    MODEL_DIR,
    "fold3_test_predictions.csv"
)

# ============================================================
# LOAD BEST MODEL
# ============================================================

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE
)

model.load_state_dict(
    checkpoint["model_state"]
)

model = model.to(DEVICE)
model.eval()

print("✓ Best readiness-only model loaded")

print(
    "Best epoch:",
    checkpoint.get(
        "epoch",
        "unknown"
    )
)

print(
    "Best validation loss:",
    checkpoint.get(
        "best_val",
        "unknown"
    )
)

# ============================================================
# TEST INFERENCE
# ============================================================

all_ids = []

true_values = []
pred_values = []

with torch.no_grad():

    for batch in tqdm(
        test_loader,
        desc="Fold 3 test"
    ):

        images = batch[
            "image"
        ].to(
            DEVICE,
            non_blocking=True
        )

        targets = batch[
            "readiness"
        ].cpu().numpy()

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            predictions = model(
                images
            )

        predictions = (
            predictions
            .float()
            .cpu()
            .numpy()
        )

        all_ids.extend(
            list(
                batch["image_id"]
            )
        )

        true_values.append(
            targets
        )

        pred_values.append(
            predictions
        )

# ============================================================
# CONCATENATE
# ============================================================

true_values = np.concatenate(
    true_values,
    axis=0
)

pred_values = np.concatenate(
    pred_values,
    axis=0
)

print(
    "\nFold 3 samples:",
    len(all_ids)
)

# ============================================================
# METRIC FUNCTION
# ============================================================

def calculate_metrics(
    y_true,
    y_pred
):

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    if (
        np.std(y_true) < 1e-12
        or
        np.std(y_pred) < 1e-12
    ):

        r = np.nan
        p = np.nan

    else:

        r, p = pearsonr(
            y_true,
            y_pred
        )

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Pearson_r": r,
        "Pearson_p": p
    }

# ============================================================
# RESULTS
# ============================================================

results = []

for i, target in enumerate(
    [
        "R_seg",
        "R_F1"
    ]
):

    m = calculate_metrics(
        true_values[:, i],
        pred_values[:, i]
    )

    results.append({
        "target": target,
        **m
    })

metrics_df = pd.DataFrame(
    results
)

# ============================================================
# SAVE METRICS
# ============================================================

metrics_df.to_csv(
    METRICS_PATH,
    index=False
)

# ============================================================
# SAVE PREDICTIONS
# ============================================================

predictions_df = pd.DataFrame({
    "image_id": all_ids,

    "R_seg_true":
        true_values[:, 0],

    "R_seg_pred":
        pred_values[:, 0],

    "R_F1_true":
        true_values[:, 1],

    "R_F1_pred":
        pred_values[:, 1]
})

predictions_df = (
    predictions_df
    .sort_values(
        "image_id",
        key=lambda x: x.astype(int)
    )
    .reset_index(drop=True)
)

predictions_df.to_csv(
    PREDICTIONS_PATH,
    index=False
)

# ============================================================
# PRINT
# ============================================================

print("\n" + "=" * 80)
print("READINESS-ONLY FOLD 3 RESULTS")
print("=" * 80)

print(
    metrics_df.round(5)
    .to_string(index=False)
)

print("\nSaved metrics:")
print(METRICS_PATH)

print("\nSaved predictions:")
print(PREDICTIONS_PATH)

print("=" * 80)

## 9. Demo: input, prediction, and Grad-CAM

Selects an image from the held-out test set, loads the best Model 2 checkpoint, and displays the input image, Grad-CAM overlays for `R_seg` and `R_F1`, and predicted versus target values. Grad-CAM highlights regions that influence each readiness output; it is not a segmentation mask.

In [ ]:

# ============================================================
# INDEPENDENT MODEL 2 DEMO — 5 EXAMPLES + GRAD-CAM
# Original | Degraded | Grad-CAM R_seg | Grad-CAM R_F1
# ============================================================

# 1. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# 2. Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import efficientnet_b0

from IPython.display import display

# ============================================================
# 3. Paths and settings
# ============================================================

PROJECT_ROOT = "/content/drive/MyDrive/AI_Readiness_Project"
DATASET_ROOT = os.path.join(PROJECT_ROOT, "AI_Readiness_Dataset (1)")

TARGET_FILE = os.path.join(
    DATASET_ROOT,
    "Targets",
    "AI_Readiness_Targets_CORRECTED.csv"
)

METADATA_FILE = os.path.join(DATASET_ROOT, "metadata.csv")

CHECKPOINT_PATH = os.path.join(
    PROJECT_ROOT,
    "Models",
    "EfficientNet_Rseg_RF1_ReadinessOnly",
    "checkpoints",
    "best_model.pth"
)

# Change this to 4 if you want four examples.
N_EXAMPLES = 5

# None = randomly select examples from the held-out test set.
# Example: DEMO_IMAGE_ID = "01234" to select a specific test image.
DEMO_IMAGE_ID = None

# Fold boundaries used by your existing demo.
FOLD1_SIZE = 2656
FOLD2_SIZE = 2523
FOLD3_SIZE = 2722
TOTAL_SIZE = FOLD1_SIZE + FOLD2_SIZE + FOLD3_SIZE

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# 4. Check required files
# ============================================================

for path in [TARGET_FILE, METADATA_FILE, CHECKPOINT_PATH]:
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Required file not found:\n{path}")

# ============================================================
# 5. Load and match metadata and targets
# ============================================================

targets = pd.read_csv(TARGET_FILE)
metadata = pd.read_csv(METADATA_FILE)

if "image_id" not in targets.columns:
    raise ValueError(
        f"'image_id' missing from target CSV. Columns: {list(targets.columns)}"
    )

if "image_name" not in metadata.columns:
    raise ValueError(
        f"'image_name' missing from metadata. Columns: {list(metadata.columns)}"
    )

# Match IDs the same way as your working one-image demo.
targets["image_id"] = (
    targets["image_id"]
    .astype(str)
    .str.replace(".png", "", regex=False)
    .str.zfill(5)
)

metadata["image_id"] = (
    metadata["image_name"]
    .astype(str)
    .str.replace(".png", "", regex=False)
    .str.zfill(5)
)

# Merge while avoiding duplicate target rows.
targets = targets.drop_duplicates(subset=["image_id"])
df = metadata.merge(targets, on="image_id", how="inner")

if df.empty:
    raise ValueError(
        "No matching image IDs found. Check image_name and image_id "
        "in metadata.csv and the target CSV."
    )

# Resolve image paths relative to the dataset root.
def resolve_path(path):
    path = str(path)

    candidates = [
        path,
        os.path.join(DATASET_ROOT, path),
        os.path.join(PROJECT_ROOT, path)
    ]

    for candidate in candidates:
        if os.path.isfile(candidate):
            return candidate

    return None

for col in ["original_path", "degraded_path"]:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

    df[col] = df[col].apply(resolve_path)

# Keep only rows with both images present.
df = df.dropna(subset=["original_path", "degraded_path"]).copy()
df = df.reset_index(drop=True)

# ============================================================
# 6. Recreate the held-out test split
# ============================================================

all_ids = (
    metadata.drop_duplicates("image_id")
    .sort_values(
        "image_id",
        key=lambda x: x.astype(int)
    )["image_id"]
    .tolist()
)

if len(all_ids) != TOTAL_SIZE:
    raise ValueError(
        f"Expected {TOTAL_SIZE} metadata rows for the specified fold "
        f"boundaries, but found {len(all_ids)}. "
        "Use the same metadata file and split boundaries as training."
    )

fold_map = {}

for i, image_id in enumerate(all_ids):
    if i < FOLD1_SIZE:
        fold_map[image_id] = "train"
    elif i < FOLD1_SIZE + FOLD2_SIZE:
        fold_map[image_id] = "validation"
    else:
        fold_map[image_id] = "test"

df["split"] = df["image_id"].map(fold_map)
test_df = df[df["split"] == "test"].copy().reset_index(drop=True)

if test_df.empty:
    raise ValueError("No held-out test images found.")

# Select up to five examples.
if DEMO_IMAGE_ID is not None:
    requested_id = str(DEMO_IMAGE_ID).replace(".png", "").zfill(5)

    selected = test_df[
        test_df["image_id"].astype(str).str.zfill(5) == requested_id
    ]

    if selected.empty:
        raise ValueError(
            f"Image ID {requested_id} is not in the held-out test split."
        )

    examples = selected.head(1).copy()

else:
    examples = test_df.sample(
        n=min(N_EXAMPLES, len(test_df)),
        random_state=42
    ).reset_index(drop=True)

print(f"Number of examples selected: {len(examples)}")
print(f"Available held-out test images: {len(test_df)}")

# ============================================================
# 7. Rebuild Model 2 and load checkpoint
# ============================================================

class ReadinessOnlyEfficientNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = efficientnet_b0(weights=None)

        feature_dim = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()

        self.head = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.BatchNorm1d(256),
            nn.SiLU(),
            nn.Dropout(0.25),
            nn.Linear(256, 2)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)

model = ReadinessOnlyEfficientNet().to(DEVICE)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

state = checkpoint.get("model_state", checkpoint)
model.load_state_dict(state, strict=True)
model.eval()

print("Model 2 checkpoint loaded successfully.")
print("Checkpoint epoch:", checkpoint.get("epoch", "Not recorded"))

# ============================================================
# 8. Image preprocessing
# ============================================================

MEAN = torch.tensor(
    [0.485, 0.456, 0.406]
).view(3, 1, 1)

STD = torch.tensor(
    [0.229, 0.224, 0.225]
).view(3, 1, 1)

def preprocess(image):
    image = image.resize(
        (224, 224),
        Image.Resampling.BILINEAR
    )

    arr = np.asarray(image).astype(np.float32) / 255.0

    tensor = torch.from_numpy(arr).permute(2, 0, 1)
    tensor = (tensor - MEAN) / STD

    return tensor.unsqueeze(0).to(DEVICE)

# Grad-CAM target layer.
target_layer = model.backbone.features[-1]

# ============================================================
# 9. Generate predictions and Grad-CAM for all examples
# ============================================================

all_results = []

fig, axes = plt.subplots(
    len(examples),
    4,
    figsize=(20, 5 * len(examples)),
    squeeze=False
)

for i, (_, row) in enumerate(examples.iterrows()):

    image_id = str(row["image_id"]).zfill(5)

    original_path = row["original_path"]
    degraded_path = row["degraded_path"]

    if not os.path.isfile(original_path):
        raise FileNotFoundError(f"Original image not found: {original_path}")

    if not os.path.isfile(degraded_path):
        raise FileNotFoundError(f"Degraded image not found: {degraded_path}")

    original_image = Image.open(original_path).convert("RGB")
    degraded_image = Image.open(degraded_path).convert("RGB")

    input_tensor = preprocess(degraded_image)

    display_input = np.asarray(
        degraded_image.resize(
            (224, 224),
            Image.Resampling.BILINEAR
        )
    )

    # Store feature activations and gradients for Grad-CAM.
    activations = {}
    gradients = {}

    def forward_hook(module, inputs, output):
        activations["feature"] = output
        output.register_hook(
            lambda grad: gradients.__setitem__("feature", grad)
        )

    hook_handle = target_layer.register_forward_hook(forward_hook)

    try:
        model.zero_grad(set_to_none=True)

        output = model(input_tensor)[0]
        predictions = output.detach().cpu().numpy()

        cams = {}

        # Index 0 = segmentation readiness; index 1 = classification readiness.
        for index, name in [(0, "R_seg"), (1, "R_F1")]:

            model.zero_grad(set_to_none=True)
            gradients.pop("feature", None)

            output[index].backward(
                retain_graph=(index == 0)
            )

            feature = activations["feature"].detach()
            grad = gradients["feature"].detach()

            weights = grad.mean(
                dim=(2, 3),
                keepdim=True
            )

            cam = torch.relu(
                (weights * feature).sum(dim=1, keepdim=True)
            )

            cam = F.interpolate(
                cam,
                size=(224, 224),
                mode="bilinear",
                align_corners=False
            )[0, 0].detach().cpu().numpy()

            if cam.max() > cam.min():
                cam = (
                    (cam - cam.min()) /
                    (cam.max() - cam.min())
                )
            else:
                cam = np.zeros_like(cam)

            cams[name] = cam

    finally:
        hook_handle.remove()

    pred_rseg = float(predictions[0])
    pred_rf1 = float(predictions[1])

    # Panel 1: Original image.
    axes[i, 0].imshow(original_image)
    axes[i, 0].set_title(f"Image {image_id}: Original")
    axes[i, 0].axis("off")

    # Panel 2: Degraded input.
    axes[i, 1].imshow(display_input)
    axes[i, 1].set_title("Degraded model input")
    axes[i, 1].axis("off")

    # Panel 3: Grad-CAM for R_seg.
    axes[i, 2].imshow(display_input)
    axes[i, 2].imshow(
        cams["R_seg"],
        cmap="jet",
        alpha=0.45,
        vmin=0,
        vmax=1
    )
    axes[i, 2].set_title(
        f"Grad-CAM R_seg\nPrediction: {pred_rseg:.4f}"
    )
    axes[i, 2].axis("off")

    # Panel 4: Grad-CAM for R_F1.
    axes[i, 3].imshow(display_input)
    axes[i, 3].imshow(
        cams["R_F1"],
        cmap="jet",
        alpha=0.45,
        vmin=0,
        vmax=1
    )
    axes[i, 3].set_title(
        f"Grad-CAM R_F1\nPrediction: {pred_rf1:.4f}"
    )
    axes[i, 3].axis("off")

    # Compare predictions with targets when present.
    for target_name, predicted in [
        ("R_seg", pred_rseg),
        ("R_F1", pred_rf1)
    ]:
        try:
            actual = float(row[target_name])
        except (KeyError, TypeError, ValueError):
            actual = np.nan

        all_results.append({
            "Image ID": image_id,
            "Output": target_name,
            "Stored target": actual,
            "Prediction": predicted,
            "Absolute error": (
                abs(actual - predicted)
                if np.isfinite(actual)
                else np.nan
            )
        })

    print(
        f"Image {image_id} | "
        f"R_seg prediction: {pred_rseg:.4f} | "
        f"R_F1 prediction: {pred_rf1:.4f}"
    )

# ============================================================
# 10. Display all examples and results table
# ============================================================

plt.suptitle(
    "EfficientNet-B0 Model 2 — Readiness Predictions and Grad-CAM",
    fontsize=17,
    fontweight="bold",
    y=1.002
)

plt.tight_layout()
plt.show()

results_df = pd.DataFrame(all_results)

print("\nPrediction vs. target comparison:")
display(results_df.round(4))

print(
    "\nGrad-CAM note: warmer regions indicate stronger positive "
    "influence on the selected model output. Grad-CAM maps are "
    "explanations of the readiness predictions, not cell segmentation masks."
)
